In [5]:
import pandas as pd

rfm = pd.read_csv("customer_rfm_segments.csv")

In [6]:
rfm.head()

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Segment,Customer_Segment
0,12346,326,12,77556.46,2,5,5,255,At Risk
1,12347,2,8,5633.32,5,4,5,545,Champions
2,12348,75,5,2019.40,3,4,4,344,Regular/Needs Attention
3,12349,19,4,4428.69,5,3,5,535,Regular/Needs Attention
4,12350,310,1,334.40,2,1,2,212,Lost/Hibernating


In [7]:
rfm['Churned'] = (rfm['Recency'] > 180).astype(int)

# Quick check on the split
rfm['Churned'].value_counts()
print(f"Churn rate: {rfm['Churned'].mean()*100:.2f}%")

Churn rate: 40.83%


In [8]:
# Features: Frequency and Monetary only (NOT Recency, since Churned is defined FROM Recency)
X = rfm[['Frequency', 'Monetary']]
y = rfm['Churned']

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

Training set size: 4702
Test set size: 1176


In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Scale features — Frequency and Monetary are on very different scales
# (Frequency: single digits, Monetary: thousands of pounds), and logistic
# regression performs better when features are on comparable scales
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

# Predictions
y_pred = model.predict(X_test_scaled)

In [11]:
# Predict churn for all customers
X_scaled = scaler.transform(X)
rfm["Churn_Prediction"] = model.predict(X_scaled)
rfm["Churn_Probability"] = model.predict_proba(X_scaled)[:, 1]

In [12]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, classification_report

print(f"Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print(f"Precision: {precision_score(y_test, y_pred):.2%}")
print(f"Recall: {recall_score(y_test, y_pred):.2%}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nFull Report:")
print(classification_report(y_test, y_pred))

Accuracy: 73.13%
Precision: 66.53%
Recall: 68.75%

Confusion Matrix:
[[530 166]
 [150 330]]

Full Report:
              precision    recall  f1-score   support

           0       0.78      0.76      0.77       696
           1       0.67      0.69      0.68       480

    accuracy                           0.73      1176
   macro avg       0.72      0.72      0.72      1176
weighted avg       0.73      0.73      0.73      1176



In [13]:
import pandas as pd

coef_df = pd.DataFrame({
    'Feature': ['Frequency', 'Monetary'],
    'Coefficient': model.coef_[0]
})
print(coef_df)

     Feature  Coefficient
0  Frequency    -3.869866
1   Monetary    -0.180594


## Revenue at Risk

In [14]:
test_results = X_test.copy()
test_results['Actual_Churned'] = y_test
test_results['Predicted_Churned'] = y_pred
test_results['Monetary'] = rfm.loc[X_test.index, 'Monetary']

# Revenue from correctly caught churners
caught_revenue = test_results[(test_results['Actual_Churned']==1) & (test_results['Predicted_Churned']==1)]['Monetary'].sum()

# Revenue from MISSED churners (false negatives) — the real cost of the model's limitations
missed_revenue = test_results[(test_results['Actual_Churned']==1) & (test_results['Predicted_Churned']==0)]['Monetary'].sum()

total_churned_revenue = caught_revenue + missed_revenue

print(f"Total historical revenue from churned customers (test set): £{total_churned_revenue:,.2f}")
print(f"Revenue from customers correctly flagged: £{caught_revenue:,.2f} ({caught_revenue/total_churned_revenue*100:.1f}%)")
print(f"Revenue from customers MISSED by the model: £{missed_revenue:,.2f} ({missed_revenue/total_churned_revenue*100:.1f}%)")

Total historical revenue from churned customers (test set): £581,042.10
Revenue from customers correctly flagged: £158,807.75 (27.3%)
Revenue from customers MISSED by the model: £422,234.35 (72.7%)
